# Lab 3: Noise and randomness

**Goals:** see how real hardware errors change results, and build a
quantum random number generator.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/qc-workshop/blob/main/notebooks/03_noise_and_randomness.ipynb)

In [ ]:
# Install pinned versions (takes about a minute in Colab; already done locally)
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0 matplotlib pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def run(qc, shots=1000, backend=sim):
    """Run a circuit and return the measurement counts."""
    return backend.run(transpile(qc, backend), shots=shots).result().get_counts()

from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# A simulator that copies the measured error rates of a real 5-qubit IBM device
noisy = AerSimulator.from_backend(FakeManilaV2())

## 3.1 Ideal versus noisy

Real qubits are disturbed by heat, stray fields and imperfect control
pulses. The "fake backend" reproduces those error rates.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

ideal_counts = run(bell, shots=4000)
noisy_counts = run(bell, shots=4000, backend=noisy)
print("ideal:", ideal_counts)
print("noisy:", noisy_counts)
plot_histogram([ideal_counts, noisy_counts], legend=["ideal", "noisy"])

The noisy run shows some `01` and `10`. Those outcomes are impossible in
theory, so every one of them is an error.

## Exercise 3.1: Does noise grow with size?

Write `success_rate(counts, good)`: the fraction of shots whose result is
in the list `good`. Then compare GHZ states of 2, 3, 4 and 5 qubits on
the noisy simulator.

In [ ]:
def success_rate(counts, good):
    # TODO: return the fraction of shots whose result is in `good`
    ...

def ghz_circuit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc

for n in range(2, 6):
    counts = run(ghz_circuit(n), shots=4000, backend=noisy)
    print(n, "qubits:", success_rate(counts, ["0" * n, "1" * n]))

More qubits and more gates mean more chances for error. This is why
today's machines are called *noisy intermediate-scale* (NISQ) devices,
and why quantum error correction is a major research area.

## 3.2 A quantum random number generator

A qubit in equal superposition gives a 0 or 1 that, on real hardware, no
one can predict, even in principle. Eight qubits give a random byte.

`memory=True` returns the result of each shot in order instead of totals.

Caveat: on a simulator the randomness comes from an ordinary
pseudo-random generator. The physics is only real on real hardware.

In [ ]:
def random_bits(n_bits):
    qc = QuantumCircuit(n_bits)
    qc.h(range(n_bits))
    qc.measure_all()
    result = sim.run(qc, shots=1, memory=True).result()
    return result.get_memory()[0]

bits = random_bits(8)
print(bits, "=", int(bits, 2))

## Exercise 3.2: Quantum dice

Write `roll_die()` that returns a fair number from 1 to 6. Three qubits
give 0 to 7, which is two too many. Throw those results away and roll
again (this is called rejection sampling); taking the result modulo 6
would make some faces more likely than others.

In [ ]:
def roll_die():
    # TODO: use random_bits(3) and reject values that don't map to 1-6
    ...

rolls = [roll_die() for _ in range(60)]
print(rolls)

Up next: the facilitator runs the Bell circuit on a real IBM quantum
computer. Compare its results with the noisy simulator above.